# 02 — Silver: current-version preparation and quality

Read approved Bronze Delta versions and rebuild Silver tables. Complete table replacement removes products, payments and discounts removed from corrected tickets. Preserve array positions, decimal amounts, dates and all quality exceptions.

Run only after Bronze COMPLETE, without concurrent writers. Failure leaves Silver RUNNING; do not continue to Gold. Historical controls remain optional original-dataset regression checks.

In [ ]:
from pyspark.sql import functions as F
from pipeline_support import load_config, PipelineContext, assert_key, assert_same_keys
CFG = load_config()
CATALOG = CFG["catalog"]
SCHEMA = CFG["schemas"]["silver"]
K = ["CheckId", "LocationId", "BusinessDate"]
EXPECTED_TICKETS = CFG["original"]["tickets"]
EXPECTED_DAYS = CFG["original"]["days"]
TOLERANCE = float(CFG["tolerance"])
RUN_ORIGINAL_REGRESSION = CFG["run_original_regression"]
ctx = PipelineContext(spark, CFG, "silver")
ctx.begin()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
def table(name): return f"{CATALOG}.{SCHEMA}.{name}"
def save(df,name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table(name))
    return spark.table(table(name))
print("Destino:", CATALOG+"."+SCHEMA, "Run:", ctx.run_id)


## Headers and dimensions

Preserve BusinessDateRaw and provenance. Invalid keys or periods stop publication.

In [ ]:
raw=ctx.read_input("workspace.applebees_bronze.checks_raw")
base=(raw.withColumnRenamed("Id","CheckId").withColumn("BusinessDateRaw",F.col("BusinessDate"))
 .withColumn("BusinessDate",F.expr("try_cast(BusinessDate AS DATE)"))
 .withColumn("source_date",F.to_date(F.regexp_extract("source_file",r"dt=(\d{4}-\d{2}-\d{2})",1))))
assert_key(base,K,"checks")
SOURCE_TICKETS = raw.count()
assert base.count() == SOURCE_TICKETS and SOURCE_TICKETS > 0, "Cabeceras perdidas o entrada vacía"
if RUN_ORIGINAL_REGRESSION:
    assert SOURCE_TICKETS == EXPECTED_TICKETS, "Regresión original: volumen distinto"
    assert base.select("BusinessDate").distinct().count() == EXPECTED_DAYS, "Regresión original: días distintos"
assert base.filter("source_date IS NULL OR BusinessDate<>source_date").limit(1).count()==0, "Fecha de ticket distinta a partición."
checks=(base.drop("ItemsSold","Payments","Discounts","Tax","Voids")
 .withColumn("open_local",F.expr("try_cast(OpenTime AS TIMESTAMP_NTZ)"))
 .withColumn("close_local",F.expr("try_cast(CloseTime AS TIMESTAMP_NTZ)")))
checks=save(checks,"checks")
for name in ["locations","items","revenue_centers","employees"]:
    df=ctx.read_input(f"workspace.applebees_bronze.{name}_raw")
    assert_key(df,["Id"],name)
    save(df,"dim_"+name)
print("Cabecera y dimensiones guardadas.")


## Expand arrays

Empty arrays produce no child rows, while headers remain. Prices are already line amounts: do not multiply by Quantity. Discounts retain scope and item_pos where applicable.

In [ ]:
def children(field,position):
    return base.select(*K,F.posexplode(field).alias(position,"record")).select(*K,position,"record.*")
def money(df,fields):
    for field in fields:
        converted=F.expr(f"try_cast(`{field}` AS DECIMAL(20,4))")
        assert df.filter(F.col(field).isNotNull() & converted.isNull()).limit(1).count()==0, f"Importe no convertible: {field}"
        df=df.withColumn(field,converted)
    return df
items_nested=children("ItemsSold","item_pos")
items=save(money(items_nested.drop("Discounts"),["GrossPrice","SoldPrice"]),"items_sold")
payments=save(money(children("Payments","payment_pos"),["Amount","Gratuity"]),"payments")
header=children("Discounts","discount_pos").withColumn("discount_scope",F.lit("ticket"))
line=(items_nested.select(*K,"item_pos",F.posexplode("Discounts").alias("discount_pos","record"))
 .select(*K,"item_pos","discount_pos","record.*").withColumn("discount_scope",F.lit("item")))
discounts=save(money(header.unionByName(line,allowMissingColumns=True),["Amount"]),"discounts")
taxes=save(money(children("Tax","tax_pos"),["Amount"]),"taxes")
voids=save(money(children("Voids","void_pos"),["Amount"]),"voids")
for df,pos,name in [(items,"item_pos","items"),(payments,"payment_pos","payments"),(taxes,"tax_pos","taxes"),(voids,"void_pos","voids")]:
    assert_key(df,K+[pos],name)
assert_key(discounts.filter("discount_scope='ticket'"),K+["discount_pos"],"ticket discounts")
assert_key(discounts.filter("discount_scope='item'"),K+["item_pos","discount_pos"],"item discounts")


## Validate array preservation

Compare Bronze array sizes with Silver row counts before calculating metrics.

In [ ]:
for field,name in [("ItemsSold","items_sold"),("Payments","payments"),("Tax","taxes"),("Voids","voids")]:
    expected=base.select(F.when(F.col(field).isNull(),0).otherwise(F.size(field)).alias("n")).agg(F.sum("n")).first()[0] or 0
    actual=spark.table(table(name)).count()
    assert actual==expected,f"Diferencia de filas en {name}: {actual} vs {expected}"
expected_header=base.select(F.when(F.col("Discounts").isNull(),0).otherwise(F.size("Discounts")).alias("n")).agg(F.sum("n")).first()[0] or 0
expected_line=items_nested.select(F.when(F.col("Discounts").isNull(),0).otherwise(F.size("Discounts")).alias("n")).agg(F.sum("n")).first()[0] or 0
assert discounts.count()==expected_header+expected_line,"Descuentos perdidos o multiplicados."
print("Arrays conservados, incluidos descuentos en ambos niveles.")

for field,name,position in [("ItemsSold","items_sold","item_pos"),("Payments","payments","payment_pos"),("Tax","taxes","tax_pos"),("Voids","voids","void_pos")]:
    expected_keys = base.select(*K,F.posexplode(field).alias(position,"record"))
    assert_same_keys(expected_keys,spark.table(table(name)),K+[position],name)
assert_same_keys(header,discounts.filter("discount_scope='ticket'"),K+["discount_pos"],"descuentos ticket")
assert_same_keys(line,discounts.filter("discount_scope='item'"),K+["item_pos","discount_pos"],"descuentos producto")


## Quality indicators

Use the monetary equation validated in EDA. Missing taxes/discounts are explicitly treated as zero. Reconciliation does not establish operational legitimacy. Item discrepancies are flagged, not corrected automatically.

In [ ]:
checks=spark.table("workspace.applebees_silver.checks")
items=spark.table("workspace.applebees_silver.items_sold")
payments=spark.table("workspace.applebees_silver.payments")
discounts=spark.table("workspace.applebees_silver.discounts")
taxes=spark.table("workspace.applebees_silver.taxes")
voids=spark.table("workspace.applebees_silver.voids")
d=discounts.groupBy(*K).agg(F.max(F.when(F.col("Amount")>0,1).otherwise(0)).alias("has_discount"),F.sum(F.when(F.col("discount_scope")=="ticket",F.col("Amount")).otherwise(0)).alias("ticket_discount_amount"),F.sum(F.when(F.col("discount_scope")=="item",F.col("Amount")).otherwise(0)).alias("item_discount_amount"))
p=payments.groupBy(*K).agg(F.count("*").alias("payment_count"),F.sum("Amount").alias("payment_amount_raw"),F.sum("Gratuity").alias("gratuity_raw"),F.max(F.when(F.col("Gratuity")<0,1).otherwise(0)).alias("has_negative_gratuity"))
i=items.groupBy(*K).agg(F.count("*").alias("item_line_count"),F.sum("GrossPrice").alias("grossprice_sum_unvalidated"),F.sum("SoldPrice").alias("soldprice_sum_unvalidated"))
v=voids.groupBy(*K).agg(F.count("*").alias("void_count"))
t=taxes.groupBy(*K).agg(F.sum("Amount").alias("tax_amount"))
facts=checks
for child in [d,p,i,v,t]:facts=facts.join(child,K,"left")
facts=facts.fillna(0,subset=["has_discount","payment_count","item_line_count","void_count","has_negative_gratuity"])
facts.createOrReplaceTempView("eda_input_facts")
for name,df in {"items":items,"payments":payments,"discounts":discounts,"taxes":taxes,"voids":voids}.items():df.createOrReplaceTempView("eda_input_"+name)
for alias,name in {"locations":"dim_locations","channels":"dim_revenue_centers","employees":"dim_employees"}.items():spark.table("workspace.applebees_silver."+name).createOrReplaceTempView("eda_input_"+alias)
spark.sql(f"CREATE OR REPLACE TEMP VIEW eda_parameters AS SELECT {TOLERANCE} AS tolerance, 100 AS min_employee_tickets,30 AS min_peer_tickets")


In [ ]:
result = spark.sql("""
CREATE OR REPLACE TEMP VIEW eda_payment_summary AS
SELECT CheckId, LocationId, BusinessDate,
 COUNT(*) AS payment_records,
 CASE WHEN COUNT(DISTINCT PayCatName) = 1 THEN MAX(PayCatName) ELSE 'MULTIPLES/O_DESCONOCIDOS' END AS payment_method,
 CASE WHEN SUM(CASE WHEN IsCreditCard = true THEN 1 ELSE 0 END) = COUNT(*) THEN 'SOLO_TARJETA'
      WHEN SUM(CASE WHEN UPPER(PayCatName) = 'CASH' THEN 1 ELSE 0 END) = COUNT(*) THEN 'SOLO_EFECTIVO'
      ELSE 'OTROS_O_MIXTOS' END AS payment_family,
 SUM(CASE WHEN Gratuity > 0 THEN 1 ELSE 0 END) AS positive_tip_records,
 SUM(CASE WHEN Amount < 0 THEN 1 ELSE 0 END) AS negative_payment_records,
 COUNT(DISTINCT CASE WHEN EmployeeId > 0 THEN EmployeeId END) AS payment_employees,
 SUM(CASE WHEN EmployeeId IS NULL OR EmployeeId <= 0 THEN 1 ELSE 0 END) AS unknown_payment_employee_records,
 MAX(CASE WHEN EmployeeId > 0 THEN EmployeeId END) AS candidate_payment_employee
FROM eda_input_payments GROUP BY CheckId, LocationId, BusinessDate
""")
display(result)


In [ ]:
result = spark.sql("""
CREATE OR REPLACE TEMP VIEW eda_void_summary AS
SELECT CheckId, LocationId, BusinessDate,
 SUM(Amount) AS void_amount_raw, SUM(Quantity) AS void_quantity_raw
FROM eda_input_voids GROUP BY CheckId, LocationId, BusinessDate
""")
display(result)


In [ ]:
result = spark.sql("""
CREATE OR REPLACE TEMP VIEW eda_tickets AS
WITH base AS (
 SELECT f.*, l.Name AS location_name, l.TimeZoneName AS location_timezone, c.Name AS channel_name,
 p.payment_method, p.payment_family, p.positive_tip_records, p.negative_payment_records,
 p.payment_employees, p.unknown_payment_employee_records,
 CASE WHEN p.payment_employees = 1 AND p.unknown_payment_employee_records = 0
      THEN p.candidate_payment_employee END AS payment_employee_id,
 COALESCE(v.void_amount_raw, 0) AS void_amount_raw, COALESCE(v.void_quantity_raw, 0) AS void_quantity_raw,
 COALESCE(f.ticket_discount_amount, 0) + COALESCE(f.item_discount_amount, 0) AS discount_total_raw,
 f.soldprice_sum_unvalidated - COALESCE(f.ticket_discount_amount, 0) AS net_sales_candidate,
 f.soldprice_sum_unvalidated - COALESCE(f.ticket_discount_amount, 0)
   + COALESCE(f.tax_amount, 0) + COALESCE(f.ServiceCharge, 0) AS account_candidate,
 (f.payment_count > 0 AND f.item_line_count > 0 AND f.payment_amount_raw IS NOT NULL
  AND f.gratuity_raw IS NOT NULL AND f.soldprice_sum_unvalidated IS NOT NULL) AS assessable
 FROM eda_input_facts f
 LEFT JOIN eda_input_locations l ON f.LocationId = l.Id
 LEFT JOIN eda_input_channels c ON f.RevenueCenterId = c.Id
 LEFT JOIN eda_payment_summary p USING (CheckId, LocationId, BusinessDate)
 LEFT JOIN eda_void_summary v USING (CheckId, LocationId, BusinessDate)
), reconciled AS (
 SELECT *, payment_amount_raw + gratuity_raw - account_candidate AS reconciliation_error
 FROM base
), classified AS (
 SELECT *,
 CASE WHEN payment_count = 0 AND item_line_count = 0 THEN 'SIN_PRODUCTOS_NI_PAGOS'
      WHEN payment_count = 0 THEN 'SIN_PAGOS'
      WHEN item_line_count = 0 THEN 'SIN_PRODUCTOS'
      WHEN NOT assessable THEN 'IMPORTES_INCOMPLETOS'
      WHEN ABS(reconciliation_error) > p.tolerance THEN 'NO_CONCILIA'
      WHEN account_candidate <= 0 THEN 'CONCILIA_CUENTA_NO_POSITIVA'
      ELSE 'CONCILIA_CUENTA_POSITIVA' END AS account_status,
 COALESCE(assessable AND ABS(reconciliation_error) <= p.tolerance, false) AS reconciled,
 (grossprice_sum_unvalidated > 0 AND discount_total_raw >= 0
  AND discount_total_raw <= grossprice_sum_unvalidated + p.tolerance) AS discount_denominator_valid
 FROM reconciled CROSS JOIN eda_parameters p
)
SELECT *,
 CASE WHEN reconciled AND discount_denominator_valid
      THEN 100.0 * TRY_DIVIDE(discount_total_raw, grossprice_sum_unvalidated) END AS discount_pct,
 CASE WHEN reconciled AND account_candidate > 0 AND positive_tip_records = 0 AND gratuity_raw <= 0
      THEN -gratuity_raw END AS tip_amount_interpreted,
 CASE WHEN reconciled AND account_candidate > 0 AND positive_tip_records = 0 AND gratuity_raw <= 0
      THEN 100.0 * TRY_DIVIDE(-gratuity_raw, account_candidate) END AS tip_pct,
 COALESCE(reconciled AND account_candidate > 0 AND positive_tip_records = 0
          AND gratuity_raw <= 0 AND payment_family = 'SOLO_TARJETA', false) AS tip_comparable
FROM classified
""")
display(result)


In [ ]:
result = spark.sql(f"""
CREATE OR REPLACE TABLE workspace.applebees_silver.item_quality USING DELTA AS
WITH d AS (
 SELECT CheckId,LocationId,BusinessDate,item_pos,SUM(Amount) AS item_discount
 FROM eda_input_discounts WHERE discount_scope='item' GROUP BY CheckId,LocationId,BusinessDate,item_pos
)
SELECT i.CheckId,i.LocationId,i.BusinessDate,i.item_pos,i.ItemId,i.EmployeeId,i.GrossPrice,i.SoldPrice,
 COALESCE(d.item_discount,0) AS item_discount,
 i.GrossPrice-i.SoldPrice-COALESCE(d.item_discount,0) AS line_error,
 (i.GrossPrice IS NOT NULL AND i.SoldPrice IS NOT NULL) AS assessable,
 COALESCE(ABS(i.GrossPrice-i.SoldPrice-COALESCE(d.item_discount,0))>{TOLERANCE},false) AS mismatch
FROM eda_input_items i LEFT JOIN d USING(CheckId,LocationId,BusinessDate,item_pos)
""")
print("Guardada: workspace.applebees_silver.item_quality")


In [ ]:
result = spark.sql("""
CREATE OR REPLACE TABLE workspace.applebees_silver.ticket_quality USING DELTA AS
WITH lines AS (
 SELECT CheckId,LocationId,BusinessDate,SUM(CASE WHEN mismatch THEN 1 ELSE 0 END) AS mismatched_item_lines,
 SUM(CASE WHEN NOT assessable THEN 1 ELSE 0 END) AS unassessable_item_lines
 FROM workspace.applebees_silver.item_quality GROUP BY CheckId,LocationId,BusinessDate
)
SELECT t.CheckId,t.LocationId,t.BusinessDate,t.assessable,t.reconciled,t.account_status,t.reconciliation_error,
 t.discount_denominator_valid,COALESCE(t.positive_tip_records,0) AS positive_tip_records,
 COALESCE(t.negative_payment_records,0) AS negative_payment_records,
 COALESCE(l.mismatched_item_lines,0) AS mismatched_item_lines,
 COALESCE(l.unassessable_item_lines,0) AS unassessable_item_lines,
 (t.payment_count=0) AS missing_payments,(t.item_line_count=0) AS missing_items,
 (t.OpenTime IS NOT NULL AND t.open_local IS NULL) AS invalid_open_time,
 (t.CloseTime IS NOT NULL AND t.close_local IS NULL) AS invalid_close_time,
 COALESCE(t.close_local<t.open_local,false) AS close_before_open
FROM eda_tickets t LEFT JOIN lines l USING(CheckId,LocationId,BusinessDate)
""")
print("Guardada: workspace.applebees_silver.ticket_quality")


In [ ]:
result = spark.sql("""
CREATE OR REPLACE TABLE workspace.applebees_silver.reference_quality USING DELTA AS
SELECT 'checks_unknown_location' AS check_name,COUNT(*) AS affected_rows FROM workspace.applebees_silver.checks c LEFT ANTI JOIN workspace.applebees_silver.dim_locations d ON c.LocationId=d.Id
UNION ALL SELECT 'checks_unknown_channel',COUNT(*) FROM workspace.applebees_silver.checks c LEFT ANTI JOIN workspace.applebees_silver.dim_revenue_centers d ON c.RevenueCenterId=d.Id
UNION ALL SELECT 'items_unknown_product',COUNT(*) FROM workspace.applebees_silver.items_sold i LEFT ANTI JOIN workspace.applebees_silver.dim_items d ON i.ItemId=d.Id
UNION ALL SELECT 'payments_unknown_employee',COUNT(*) FROM workspace.applebees_silver.payments p LEFT ANTI JOIN workspace.applebees_silver.dim_employees d ON p.EmployeeId=d.Id
UNION ALL SELECT 'discounts_unknown_employee',COUNT(*) FROM workspace.applebees_silver.discounts p LEFT ANTI JOIN workspace.applebees_silver.dim_employees d ON p.EmployeeId=d.Id
""")
print("Guardada: workspace.applebees_silver.reference_quality")


In [ ]:
result = spark.sql("""
SELECT account_status,COUNT(*) AS tickets FROM workspace.applebees_silver.ticket_quality GROUP BY account_status ORDER BY tickets DESC
""")
display(result)


In [ ]:
result = spark.sql("""
SELECT SUM(CASE WHEN mismatch THEN 1 ELSE 0 END) AS mismatches,COUNT(*) AS item_lines FROM workspace.applebees_silver.item_quality
""")
display(result)


In [ ]:
quality = spark.table(table("ticket_quality"))
assert_key(quality,K,"ticket_quality")
assert_same_keys(base,spark.table(table("checks")),K,"Bronze vs Silver checks")
assert_same_keys(base,quality,K,"Bronze vs Silver calidad")
assert quality.count() == SOURCE_TICKETS, "Tickets perdidos en controles"
if RUN_ORIGINAL_REGRESSION:
    assert quality.count() == EXPECTED_TICKETS, "Regresión original: tickets distintos"
    assert quality.filter("assessable AND NOT reconciled").count() == CFG["original"]["non_reconciled_tickets"], "Regresión original: no conciliados distintos"
    assert spark.table(table("item_quality")).filter("mismatch").count() == CFG["original"]["mismatched_item_lines"], "Regresión original: líneas discrepantes distintas"
print("Silver validada contra la entrada vigente; excepciones conservadas.")


## Keys and Catalog relationships

These cells run after tables are written and checked. They declare informational PK/FK constraints without RELY and retain pipeline controls. Primary keys require NOT NULL columns: nulls and uniqueness are checked before changing that property. Foreign keys require compatible types and matching parent records. Unmatched optional references, including EmployeeId/ManagerId=0, are reported and prevent declaring that whole-table FK. Employees are neither deleted nor reassigned. NULL represents absence and needs no parent.

Invalid keys or mandatory relationships stop execution. Unresolved optional relationships remain visible. Reruns recognize existing constraints without deleting them. Rebuilds require rerunning relationship blocks; rebuilding Silver also requires rebuilding Gold and its constraints. Bronze retains inputs without adding relational constraints.


In [ ]:
import re
from pyspark.sql import functions as F
relation_log = []
def ident(value):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), f"Identificador inesperado: {value}"
    return f"`{value}`"
def qualified(schema,name):
    return ".".join(ident(x) for x in ["workspace",schema,name])
def columns_sql(columns): return ", ".join(ident(x) for x in columns)
def existing_constraints(schema,name):
    return spark.sql(f"""SELECT constraint_name,constraint_type
    FROM workspace.information_schema.table_constraints
    WHERE table_catalog='workspace' AND table_schema='{schema}' AND table_name='{name}'""").collect()
def pk_columns(schema,name,constraint):
    return [r.column_name for r in spark.sql(f"""SELECT column_name FROM workspace.information_schema.key_column_usage
    WHERE constraint_catalog='workspace' AND constraint_schema='{schema}' AND constraint_name='{constraint}'
    AND table_schema='{schema}' AND table_name='{name}' ORDER BY ordinal_position""").collect()]
def add_pk(schema,name,columns,required=True):
    full=qualified(schema,name)
    df=spark.table(full)
    if df.filter(" OR ".join(f"{ident(c)} IS NULL" for c in columns)).limit(1).count():
        relation_log.append((schema,name,'PK','NO_DECLARADA: columnas nulas',','.join(columns)))
        if required: raise ValueError(f"Llave incompleta en {full}; investigar antes de declarar PK.")
        return False
    if df.groupBy(*columns).count().filter('count>1').limit(1).count():
        raise ValueError(f"Llave repetida en {full}; investigar antes de declarar PK.")
    pks=[r for r in existing_constraints(schema,name) if r.constraint_type=='PRIMARY KEY']
    if pks:
        assert pk_columns(schema,name,pks[0].constraint_name)==columns,f"PK existente diferente en {full}."
        relation_log.append((schema,name,'PK','YA_EXISTE',','.join(columns)))
        return True
    fields={f.name:f for f in df.schema.fields}
    for column in columns:
        if fields[column].nullable:
            spark.sql(f"ALTER TABLE {full} ALTER COLUMN {ident(column)} SET NOT NULL")
    constraint=f"pk_{schema}_{name}"
    spark.sql(f"ALTER TABLE {full} ADD CONSTRAINT {ident(constraint)} PRIMARY KEY ({columns_sql(columns)}) NOT ENFORCED")
    relation_log.append((schema,name,'PK','DECLARADA',','.join(columns)))
    return True

def add_fk(schema,name,columns,parent_schema,parent_name,parent_columns,label,required=False):
    full=qualified(schema,name);parent=qualified(parent_schema,parent_name)
    assert len(columns)==len(parent_columns)
    left=spark.table(full);right=spark.table(parent)
    parent_pks=[r for r in existing_constraints(parent_schema,parent_name) if r.constraint_type=='PRIMARY KEY']
    assert parent_pks and pk_columns(parent_schema,parent_name,parent_pks[0].constraint_name)==parent_columns, f"Falta PK compatible en {parent}. Ejecuta primero relaciones Silver."
    child_types={f.name:f.dataType for f in left.schema.fields};parent_types={f.name:f.dataType for f in right.schema.fields}
    assert all(child_types[c]==parent_types[p] for c,p in zip(columns,parent_columns)), f"Tipos distintos para FK de {full} hacia {parent}; revisar, no convertir automáticamente."
    # Require every fully non-null key combination to match a parent. NULL values represent missing references.
    observed=left.select(*columns).filter(" AND ".join(f"{ident(c)} IS NOT NULL" for c in columns))
    parents=right.select(*[F.col(p).alias(c) for c,p in zip(columns,parent_columns)])
    missing=observed.join(parents,columns,'left_anti').limit(1).collect()
    target=f"{parent_schema}.{parent_name}({','.join(parent_columns)})"
    if missing:
        relation_log.append((schema,name,'FK','NO_DECLARADA: referencia sin correspondencia',label+' -> '+target+'; ejemplo '+str(missing[0].asDict())))
        if required: raise ValueError(f"Referencia inválida de {full} hacia {parent}: {missing[0]}")
        return False
    constraint=f"fk_{schema}_{name}_{label}"
    if any(r.constraint_name==constraint and r.constraint_type=='FOREIGN KEY' for r in existing_constraints(schema,name)):
        relation_log.append((schema,name,'FK','YA_EXISTE',label+' -> '+target))
        return True
    spark.sql(f"ALTER TABLE {full} ADD CONSTRAINT {ident(constraint)} FOREIGN KEY ({columns_sql(columns)}) REFERENCES {parent} ({columns_sql(parent_columns)}) NOT ENFORCED")
    relation_log.append((schema,name,'FK','DECLARADA',label+' -> '+target))
    return True

def show_relations():
    display(spark.createDataFrame(relation_log,'schema STRING, table_name STRING, kind STRING, status STRING, detail STRING'))
    display(spark.sql("""SELECT table_schema,table_name,constraint_name,constraint_type
      FROM workspace.information_schema.table_constraints
      WHERE table_schema IN ('applebees_silver','applebees_gold') AND constraint_type IN ('PRIMARY KEY','FOREIGN KEY')
      ORDER BY table_schema,table_name,constraint_type"""))


In [ ]:
S='applebees_silver'
K=['CheckId','LocationId','BusinessDate']
line_discounts=spark.table('workspace.applebees_silver.discounts').filter("discount_scope='item'")
assert line_discounts.filter('item_pos IS NULL').limit(1).count()==0, 'Descuento de producto sin posición.'
assert line_discounts.select(*(K+['item_pos'])).join(spark.table('workspace.applebees_silver.items_sold').select(*(K+['item_pos'])),K+['item_pos'],'left_anti').limit(1).count()==0, 'Descuento de producto sin línea vendida.'
assert spark.table('workspace.applebees_silver.discounts').filter("discount_scope NOT IN ('item','ticket') OR discount_scope IS NULL OR (discount_scope='ticket' AND item_pos IS NOT NULL)").limit(1).count()==0, 'Nivel/posición de descuento inesperado.'

for name in ['dim_locations','dim_items','dim_revenue_centers','dim_employees']:
    add_pk(S,name,['Id'])
add_pk(S,'checks',K)
for name,position in [('items_sold','item_pos'),('payments','payment_pos'),('taxes','tax_pos'),('voids','void_pos'),('item_quality','item_pos')]:
    add_pk(S,name,K+[position])
add_pk(S,'ticket_quality',K)
add_pk(S,'reference_quality',['check_name'])
for name in ['items_sold','payments','discounts','taxes','voids','ticket_quality']:
    add_fk(S,name,K,S,'checks',K,'ticket',required=True)
add_fk(S,'item_quality',K+['item_pos'],S,'items_sold',K+['item_pos'],'item_line',required=True)
add_fk(S,'checks',['LocationId'],S,'dim_locations',['Id'],'location')
add_fk(S,'checks',['RevenueCenterId'],S,'dim_revenue_centers',['Id'],'channel')
add_fk(S,'items_sold',['ItemId'],S,'dim_items',['Id'],'product')
add_fk(S,'items_sold',['RevenueCenterId'],S,'dim_revenue_centers',['Id'],'channel')
add_fk(S,'dim_employees',['LocationId'],S,'dim_locations',['Id'],'location')
add_fk(S,'dim_employees',['HomeLocationId'],S,'dim_locations',['Id'],'home_location')
add_fk(S,'voids',['ItemId'],S,'dim_items',['Id'],'product')
for name in ['items_sold','payments','discounts','voids']:
    add_fk(S,name,['EmployeeId'],S,'dim_employees',['Id'],'employee')
for name in ['discounts','voids']:
    add_fk(S,name,['ManagerId'],S,'dim_employees',['Id'],'manager')
relation_log.append((S,'discounts','RELACION_CONDICIONAL','VALIDADA_EN_DATOS: no declarada como FK de toda la tabla','scope=item -> items_sold por '+','.join(K+['item_pos'])+'; scope=ticket no apunta a una línea.'))
relation_log.append((S,'discounts','PK','NO_DECLARADA: dos niveles con item_pos nulo en ticket','Unicidad comprobada por separado para ticket e item en 02_silver_preparacion. FK al ticket sí declarada.'))
show_relations()


In [ ]:
ctx.finish([table(name) for name in [
    "checks","dim_locations","dim_items","dim_revenue_centers","dim_employees",
    "items_sold","payments","discounts","taxes","voids",
    "item_quality","ticket_quality","reference_quality"]])
